# 进阶实践项目参考答案 05：空间表达超分辨率：消融、空间留出与守恒

把图像、粗尺度表达和细尺度目标分开比较，确认模型改进来自哪一种信息。

Kaggle 中先复制到自己的账户，再按任务顺序完成。题目只保留关键填写位置，数据读取、绘图和保存框架已经给出。

## 任务
1. 建立空间连续留出
2. 实现插值和表达基线
3. 加入 H&E 特征完成消融
4. 计算像素与聚合指标
5. 解释平滑预测的局限

In [1]:
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
SEED=42; OUT=Path('advanced05_results'); OUT.mkdir(exist_ok=True); paths=list(Path('/kaggle/input').rglob('kydw-try-a05-paired-patches.npz'))+list(Path('.').rglob('kydw-try-a05-paired-patches.npz')); assert paths,'需要课程 NPZ'
d=np.load(paths[0],allow_pickle=True); he=d['he'].astype('float32')/255; lr=d['lr'].astype('float32')/64; hr=d['hr'].astype('float32'); split=d['split'].astype(str); tr=split=='train'; te=split=='test'
base=lr[:,0]; target=hr[:,0]; he_gray=he.mean(1); height,width=target.shape[-2:]; X=np.stack([base,he_gray,he[:,0]-he[:,2]],axis=-1).reshape(-1,3); yy=target.reshape(-1); train_pixels=np.repeat(tr,height*width); test_pixels=np.repeat(te,height*width)
rng=np.random.default_rng(SEED); ids=np.where(train_pixels)[0]; ids=rng.choice(ids,min(120000,len(ids)),replace=False); model=Ridge(alpha=2).fit(X[ids],yy[ids]); pred=model.predict(X[test_pixels]).reshape((-1,height,width)).clip(0); truth=target[te]; baseline=base[te]
def corr(a,b): return float(np.corrcoef(a.ravel(),b.ravel())[0,1])
def aggregate(a): return a.reshape(len(a),height//8,8,width//8,8).sum((2,4))
result={'mae_interpolation':float(np.abs(baseline-truth).mean()),'mae_multimodal':float(np.abs(pred-truth).mean()),'pearson_interpolation':corr(baseline,truth),'pearson_multimodal':corr(pred,truth),'aggregation_mae':float(np.abs(aggregate(pred)-aggregate(lr[te,0])).mean())}
fig,ax=plt.subplots(1,5,figsize=(14,3)); vals=[np.moveaxis(he[te][0],0,-1),baseline[0],truth[0],pred[0],np.abs(pred[0]-truth[0])]; titles=['H&E','interpolation','reference','multimodal','absolute error']; [ax[i].imshow(v,cmap=None if i==0 else 'magma') for i,v in enumerate(vals)]; [ax[i].set_title(titles[i]) or ax[i].axis('off') for i in range(5)]; fig.tight_layout(); fig.savefig(OUT/'advanced05_summary.png',dpi=150); plt.close(fig); (OUT/'advanced05_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8'); print(result)


{'mae_interpolation': 0.19902850687503815, 'mae_multimodal': 0.20430897176265717, 'pearson_interpolation': 0.4877102905927163, 'pearson_multimodal': 0.500215720380498, 'aggregation_mae': 1.2503628730773926}
